# DispatchR — Colab Debug Training

This notebook runs **debug-friendly GRPO training** for the DispatchR emergency dispatch environment.

**Features:**
- Uses **Unsloth** for 2-4× faster training (auto-falls back to standard transformers)
- **Verbose debug output** — see exactly what the model generates and how it acts
- Runs on Google Colab **T4 GPU** (or Lightning AI L4)

**Setup time:** ~3 minutes (model download)
**Training time:** ~5-10 min for 10 episodes (debug mode)

## 1. Setup — Install Dependencies

In [ ]:
# Install Unsloth (fast 4-bit training)
!pip install unsloth -q

# Install remaining dependencies
!pip install transformers accelerate peft datasets -q
!pip install networkx matplotlib jmespath pydantic -q

print("\n[OK] Dependencies installed")

## 2. Clone Repo & Import

In [ ]:
import os

# Clone repo if not already present
if not os.path.exists("dead-air"):
    !git clone https://github.com/garg-tejas/dead-air.git
    %cd dead-air
else:
    %cd dead-air
    !git pull origin main

import sys
sys.path.insert(0, ".")

print("[OK] Repo ready")

## 3. Verify GPU

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[WARN] No GPU detected — training will be very slow")

## 4. Run Debug Training

This runs a small number of episodes with **verbose output** so you can see exactly what the model is doing.

**Key parameters:**
- `--episodes 10` — Number of episodes to run
- `--debug-steps 3` — Print verbose output for first 3 steps of first 3 episodes
- `--model unsloth/Qwen3.5-2B` — Model to use (4-bit via Unsloth)

**What to look for:**
- `COMPLETION:` — What the model actually generated
- `PARSED ACTION:` — What action we extracted from the completion
- `EPISODE N REWARD:` — Final reward (should be > 0 if episode completed)

In [ ]:
!python colab_train.py \
    --model unsloth/Qwen3.5-2B \
    --episodes 10 \
    --max-steps 25 \
    --debug-steps 3 \
    --difficulty learning \
    --output-dir ./outputs/colab_debug

## 5. Analyze Results

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load results
with open("./outputs/colab_debug/results.json", "r") as f:
    results = json.load(f)

rewards = results["rewards"]

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(rewards, marker="o", linewidth=1, markersize=4)
plt.axhline(y=np.mean(rewards), color="r", linestyle="--", label=f"Mean: {np.mean(rewards):.3f}")
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("Episode Rewards")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(rewards, bins=20, edgecolor="black", alpha=0.7)
plt.axvline(x=np.mean(rewards), color="r", linestyle="--", linewidth=2, label=f"Mean: {np.mean(rewards):.3f}")
plt.xlabel("Reward")
plt.ylabel("Count")
plt.title("Reward Distribution")
plt.legend()

plt.tight_layout()
plt.show()

print(f"\nSummary:")
print(f"  Episodes: {len(rewards)}")
print(f"  Mean reward: {np.mean(rewards):.4f}")
print(f"  Std reward:  {np.std(rewards):.4f}")
print(f"  Max reward:  {np.max(rewards):.4f}")
print(f"  Min reward:  {np.min(rewards):.4f}")
print(f"  Non-zero:    {sum(1 for r in rewards if r > 0)}/{len(rewards)}")

## 6. Inspect Model Outputs (Debug Logs)

Look at the detailed logs to understand what the model is generating and why.

In [ ]:
# Load detailed logs
with open("./outputs/colab_debug/logs.json", "r") as f:
    logs = json.load(f)

# Show first episode, first 3 steps
ep = logs[0]
print(f"Episode {ep['episode'] + 1} | Reward: {ep['reward']}")
print("=" * 60)

for step in ep["steps"][:3]:
    print(f"\n--- Step {step['step']} ---")
    comp = step['completion']
    print(f"Completion: {comp[:150]}{'...' if len(comp) > 150 else ''}")
    print(f"Parsed: {step['parsed_action']}")
    print(f"Result: {step['result'][:100]}{'...' if len(step['result']) > 100 else ''}")

# Show action distribution across all episodes
from collections import Counter
actions = [s["parsed_action"]["action_type"] for ep in logs for s in ep["steps"]]
print(f"\n\nAction distribution across all steps:")
for act, count in Counter(actions).most_common():
    print(f"  {act}: {count}")

## 7. Scale Up (Optional)

Once you've verified the debug run works, scale to more episodes:

In [ ]:
# Run 50 episodes with less verbose output
!python colab_train.py \
    --model unsloth/Qwen3.5-2B \
    --episodes 50 \
    --max-steps 25 \
    --debug-steps 0 \
    --difficulty learning \
    --output-dir ./outputs/colab_50ep

## Troubleshooting

| Issue | Fix |
|-------|-----|
| `No module named 'unsloth'` | Re-run cell 1 (install dependencies) |
| `CUDA out of memory` | Reduce `--max-steps` or use smaller model (`unsloth/Qwen3.5-0.8B`) |
| `reward = 0` for all episodes | Normal for small test runs — need >20 episodes for episodes to complete |
| Model outputs gibberish | Check that `unsloth/Qwen3.5-2B` loaded (not Base model) |
| `ImportError: cannot import name 'DeadAirGRPOEnv'` | Make sure you're in `/content/dead-air` directory |

## Next Steps

1. **Verify debug run** — Check that completions contain parseable actions
2. **Scale to 200 episodes** — Full training run
3. **Generate plots** — Use `plot_rewards.py`
4. **Save to HF** — Push model and environment to HuggingFace